## 2.3 文本预处理 - One-hot 表示 & 为什么需要 Embedding

#### 1、为什么这一小节要紧接着学习

##### 1.1 上一小节我们只完成了“编号”，还没有真正完成“表示”
上一小节我们已经知道：

- 文本先分词，得到 token
- 再建立词表 `vocabulary`
- 然后把 token 映射成 id

例如：

- `I → 2`
- `love → 3`
- `AI → 4`

于是句子：

`I / love / AI`

就可以表示成：

`2 / 3 / 4`

但是这里有一个非常重要的问题：

这些 `2`、`3`、`4` 真的能直接代表单词本身吗？

答案是：不能。

因为这些数字目前只是“编号”，不是“表示”。

这就像给每个学生分配了学号，但学号本身并不能描述这个学生的性格、能力和特点。

所以，接下来我们就要学习：

单词在进入 RNN 之前，到底应该被表示成什么形式。

##### 1.2 这一小节是在解决“数字 id 还不够”的问题
很多初学者会以为：

既然 token 已经变成数字了，那就已经可以直接送进模型了。

其实这只是完成了“身份标记”，还没有完成“可学习的特征表示”。

因为模型真正擅长处理的，不是随便几个离散编号，而是能表达信息的数值向量。

所以这一节的核心问题是：

- 为什么 id 不够
- 什么是 one-hot
- 为什么 one-hot 也有问题
- 为什么最后还要用 Embedding

##### 1.3 这一节是后面理解 Embedding 层的前提
如果你不先理解：

- id 为什么不行
- one-hot 为什么出现
- one-hot 为什么又不够好

那么后面学习 Embedding 时，很容易变成机械记忆。

所以这一小节的任务，就是把这条逻辑链彻底打通。

#### 2、为什么 token id 不能直接当作单词表示

##### 2.1 id 只是编号，不代表语义
例如词表中有：

- `I → 2`
- `love → 3`
- `AI → 4`
- `hate → 5`

这些数字只是编号。

你不能因为：

- `hate = 5`
- `love = 3`

就认为 `hate` 比 `love` “更大”。

也不能因为：

- `AI = 4`
- `love = 3`

就认为 `AI` 和 `love` 比较接近。

这些 id 没有语义距离，它们只是标签。

##### 2.2 模型会误把 id 当成有大小关系的数值
如果我们直接把 id 输入模型，例如：

`I love AI`  
$\rightarrow$  
`2 3 4`

那么模型会看到：

`2`、`3`、`4` 这些数字。

而神经网络天然会把数值理解为带有某种大小和距离关系的量。

例如它可能会“误以为”：

- `4` 比 `3` 大一点
- `5` 比 `2` 远很多

但这在词 id 中根本没有意义。

因为：

- `love = 3`
- `AI = 4`
- `hate = 5`

这里只是我们随手分配的身份编号，不是自然存在的数学值。

##### 2.3 所以我们需要“重新表示”单词
也就是说：

`token → id`

这一步只是为了查找方便，便于建立统一索引。

但真正送进模型前，我们还要进一步把每个单词转换成一种更合理的向量形式。

这就是后面要引出的：

- one-hot 表示
- Embedding 表示


#### 3、什么是 one-hot 表示

##### 3.1 one-hot 的基本思想
one-hot 是一种最基础的离散符号表示方法。

它的核心思想是：

如果词表大小是 $V$，那么每个 token 就用一个长度为 $V$ 的向量表示。

这个向量中只有一个位置是 $1$，其他位置全是 $0$。

所以它叫 one-hot：

- `one` = 只有一个 `1`
- `hot` = 这个位置被“点亮”

##### 3.2 一个最简单的例子
假设词表如下：

- `I → 0`
- `love → 1`
- `AI → 2`
- `hate → 3`

那么词表大小 $V = 4$。

这时每个单词就可以表示为长度为 $4$ 的向量：

`I`  
$\rightarrow$  
`[1, 0, 0, 0]`

`love`  
$\rightarrow$  
`[0, 1, 0, 0]`

`AI`  
$\rightarrow$  
`[0, 0, 1, 0]`

`hate`  
$\rightarrow$  
`[0, 0, 0, 1]`

你会发现：

每个单词都有自己唯一的位置。

##### 3.3 one-hot 本质上是在做“位置标记”
可以把 one-hot 理解成：

“我是谁，就把我对应的位置点亮”

例如：

`love` 对应 `id = 1`

那么它的 one-hot 向量就在第 `1` 个位置放 `1`，其余位置全部放 `0`：

`[0, 1, 0, 0]`

所以 one-hot 其实是：

把“编号”变成了“向量形式的编号表示”。

#### 5、为什么 one-hot 比直接用 id 更合理

##### 4.1 one-hot 避免了 id 的大小误导
如果直接用 id：

- `love = 1`
- `AI = 2`
- `hate = 3`

模型可能误以为 `3` 比 `1` 大很多。

但如果改成 one-hot：

`love`  
$\rightarrow$  
`[0, 1, 0, 0]`

`AI`  
$\rightarrow$  
`[0, 0, 1, 0]`

`hate`  
$\rightarrow$  
`[0, 0, 0, 1]`

那么不同单词之间不再表现为“大小关系”，而是“不同位置被激活”。

这就更合理了。

##### 4.2 one-hot 让每个词有了向量形式
RNN 和其他神经网络更喜欢处理向量。

所以相较于一个单独的整数 id，one-hot 至少已经把单词变成了一个向量。

这就为后续神经网络处理提供了更自然的输入形式。

##### 4.3 one-hot 明确区分了不同词
在 one-hot 中：

每个词对应唯一一个位置。

所以不同词不会混淆。

例如：

`I`  
$\rightarrow$  
`[1, 0, 0, 0]`

`love`  
$\rightarrow$  
`[0, 1, 0, 0]`

这两个词的表示完全不同，界限非常清楚。

#### 5、一个句子如何表示成 one-hot 序列

##### 5.1 先看句子
句子：

`I love AI`

如果词表是：

- `I → 0`
- `love → 1`
- `AI → 2`
- `hate → 3`

##### 5.2 每个 token 都可以转成 one-hot
`I`  
$\rightarrow$  
`[1, 0, 0, 0]`

`love`  
$\rightarrow$  
`[0, 1, 0, 0]`

`AI`  
$\rightarrow$  
`[0, 0, 1, 0]`

##### 5.3 整句话就可以表示成一个向量序列
可以写成：

```python
[
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0]
]
```

你可以把它理解为：

这句话有 $3$ 个时间步，每个时间步输入一个 one-hot 向量。

这时它终于开始接近 RNN 可处理的输入形式了。

#### 6、one-hot 的本质缺陷是什么

虽然 one-hot 比直接用 id 更合理，但它仍然有很大的问题。

这一部分非常关键。

##### 6.1 缺陷一：向量非常稀疏 sparse
one-hot 向量中：

- 只有一个位置是 `1`
- 其他位置全是 `0`

例如：

`[0, 0, 0, 0, 1, 0, 0, 0, 0, 0]`

如果词表只有几百个词，这还不算什么。

但如果词表有 `10000` 个、`50000` 个甚至更多词，那么每个向量都会非常长，而且几乎全是 `0`。

这就叫稀疏表示 `sparse representation`。

它的问题是：

- 占空间
- 计算效率不高
- 信息利用率低

##### 6.2 缺陷二：维度会随着词表变得非常大
one-hot 的维度永远等于词表大小。

如果词表大小是：

`10000`

那么每个词的 one-hot 向量长度就是：

`10000`

如果词表大小是：

`50000`

那每个向量长度就是：

`50000`

这会让输入维度非常夸张。

而在真实 NLP 任务中，词表往往很大，所以 one-hot 很不经济。

##### 6.3 缺陷三：one-hot 无法表达词与词之间的相似性
这是最核心的缺陷。

例如下面这些词：

- `good`
- `great`
- `excellent`

从语义上说，它们都比较接近，都是正向词。

但在 one-hot 中，它们可能分别是：

`good`  
$\rightarrow$  
`[1, 0, 0, 0, 0, 0]`

`great`  
$\rightarrow$  
`[0, 1, 0, 0, 0, 0]`

`excellent`  
$\rightarrow$  
`[0, 0, 1, 0, 0, 0]`

你会发现，这三个向量之间一样“远”。

它们彼此没有体现出任何语义接近关系。

再比如：

- `cat` 和 `dog`

它们都属于动物，语义上比较接近。

但 one-hot 完全看不出来。

也就是说：

one-hot 只能区分“是不是同一个词”，却无法表达“两个词像不像”。

##### 6.4 缺陷四：模型无法直接从 one-hot 中得到压缩的语义特征
one-hot 只是“点亮某个位置”，它没有主动把语义压缩成更有信息密度的低维表示。

换句话说：

它只是身份标记，不是高质量语义表示。

所以如果我们希望模型更高效地学习词义关系，就需要一种更紧凑、更有语义能力的表示方法。

>这就引出了 Embedding。

#### 7、为什么我们需要 Embedding

##### 7.1 Embedding 的目标：把离散词变成稠密向量
Embedding 的核心思想是：

把每个 token 映射成一个低维、稠密、可训练的向量。

例如：

`love`  
$\rightarrow$  
`[0.12, -0.57, 0.83, 0.21]`

`good`  
$\rightarrow$  
`[0.88, 0.34, -0.12, 0.49]`

这种向量和 one-hot 不一样：

- 它不是几乎全 `0`
- 它维度通常更低
- 它是模型可以学习和调整的

所以 Embedding 是一种更适合神经网络学习的表示。

##### 7.2 Embedding 能把高维稀疏表示压缩成低维稠密表示
例如：

词表大小 $= 10000$

如果用 one-hot，每个词要用长度 `10000` 的向量表示。

但如果用 Embedding，我们可以把每个词表示成长度 `128` 的向量。

也就是说：

`10000` 维  
$\rightarrow$  
`128` 维

这样会更紧凑，也更高效。

##### 7.3 Embedding 可以学习语义相似性
这是最重要的一点。

在 Embedding 空间中，如果两个词经常出现在相似上下文里，它们的向量就可能学得比较接近。

例如：

- `good` 和 `great`
- `cat` 和 `dog`
- `king` 和 `queen`

这些词的向量可能在某种程度上彼此接近。

这就让模型真正开始拥有“语义建模能力”。

而这是 one-hot 完全做不到的。

##### 7.4 Embedding 更适合作为 RNN 的输入
RNN 每个时间步需要输入一个数值向量。

one-hot 虽然也是向量，但通常太高维、太稀疏、语义能力太弱。

Embedding 向量则：

- 维度更合理
- 信息更密集
- 可训练
- 更能表达词义关系

所以在实际深度学习中，文本进入 RNN 前，通常都会先经过 Embedding 层。


#### 8、one-hot 和 Embedding 的关系是什么

##### 8.1 one-hot 不是错的，而是“基础但不够好”
一定要理解：

one-hot 并不是错误表示。

它在理论上是合理的，也很有教学价值。

因为它告诉我们：

每个离散 token 都可以先变成一个向量形式的身份表示。

但问题在于：

它太稀疏、太高维、没有语义结构。

所以在实际模型中，我们通常不会停留在 one-hot。

##### 8.2 可以把 Embedding 看成 one-hot 的升级版
从理解上，你可以把 Embedding 想成：

在保留“每个词都有独立身份”的基础上，进一步把词表示成更紧凑、更有语义的向量。

所以逻辑链可以记成：

`token`  
$\rightarrow$ `id`  
$\rightarrow$ one-hot 的思想  
$\rightarrow$ 发现 one-hot 不够好  
$\rightarrow$ 改用 Embedding

##### 8.3 从实现角度看，Embedding 常常像“查表”
这一点先简单建立印象。

假设词表大小是 `10000`，Embedding 维度是 `128`，那么 Embedding 层本质上像一个矩阵：

$10000 \times 128$

也就是：

- `10000` 行
- `128` 列

每个词 id 对应其中一行。

所以如果：

`love` 的 id $= 3$

那么 Embedding 层就取出第 `3` 行这个 `128` 维向量，作为 `love` 的表示。

这也是为什么前面必须先有词表和 id。